In [ ]:
from kloppy import sportec, statsbomb
from kloppy.domain.models import PitchDimensions, Dimension
from databallpy import get_game_from_kloppy

print("Parsing local XML tracking and JSON event files via Kloppy...")

# 1. Load Local Sportec Tracking Data
tracking_dataset = sportec.load_tracking(
    raw_data="hdfc_tracking_event_matches/tracking/positional_data_raw/MLS-COM-000001_MLS-SEA-0001KA_MLS-MAT-0009B7.xml",
    meta_data="hdfc_tracking_event_matches/tracking/match_information/MLS-COM-000001_MLS-SEA-0001KA_MLS-MAT-0009B7.xml",
    coordinates="sportec" 
)

# 3. Load Local StatsBomb Event Data 
event_dataset = statsbomb.load(
    event_data="hdfc_tracking_event_matches/events/4037572.json",
    lineup_data="hdfc_tracking_event_matches/lineups/4037572.json"
)

# --- FIX: Define an identical custom pitch dimensions structure for both ---
# Rather than manually creating PitchDimensions, we can copy the dimensions 
# from the StatsBomb dataset OR force both to use StatsBomb's exact coordinate system.
# The safest approach for databallpy is transforming BOTH to ensure metadata parity.

tracking_dataset = tracking_dataset.transform(to_coordinate_system="statsbomb")
event_dataset = event_dataset.transform(to_coordinate_system="statsbomb")

# Explicitly align the pitch metadata attributes so databallpy's strict check passes
tracking_dataset.metadata.pitch_dimensions = event_dataset.metadata.pitch_dimensions

print("Rescaled both datasets to identical coordinate systems and metadata.")

# 4. Transform and combine into a DataBallPy Game object
game = get_game_from_kloppy(
    tracking_dataset=tracking_dataset, 
    event_dataset=event_dataset
)

print("\n--- MATCH COMBINATION COMPLETED ---")
print(f"Tracking Dataset Frames: {len(game.tracking_data)}")
print(f"Event Dataset Logs: {len(game.event_data)}")

In [ ]:
# Sync the event and tracking data
game.synchronise_tracking_and_event_data()
# Combined data can be accessed via game.tracking_data and game.event_data DataFrames
print("\n--- SYNCHRONIZATION COMPLETED ---")
# Overall sync certainty
print(f"Sync Certainty: {game.tracking_data['sync_certainty'].mean():.4f}")
# Print certainty variance to check for consistency
print(f"Sync Certainty Variance: {game.tracking_data['sync_certainty'].var():.6f}")

In [ ]:
import itertools
import numpy as np
import pandas as pd
# Import the module where the function lives
import databallpy.utils.synchronise_tracking_and_event_data as sync_module 

# Save a reference to the original function so we don't break anything permanently
original_combine_cost_functions = sync_module.combine_cost_functions

# --- STEP 1: Define your Grid Search Space ---
# Adjust these lists to the specific ranges you want to test
grid_space = {
    "time_cost": [0.5, 1.0, 1.5],
    "ball_event_dist_cost": [1.0, 1.5, 2.0],
    "ball_player_dist_cost": [1.0], # keeping some static reduces total iterations
    "ball_acc_cost": [1.0],
    "player_ball_dist_inc_cost": [1.0],
    "goal_angle_cost": [1.0]
}

# Generate all combinations
keys, values = zip(*grid_space.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total grid search iterations to run: {len(experiments)}")

# --- STEP 2: Run the Grid Search ---
results = []

for i, custom_weights in enumerate(experiments):
    print(f"Running iteration {i+1}/{len(experiments)} with weights: {custom_weights}")
    
    # Define a wrapper function that forces your custom weights
    def mocked_combine_cost_functions(costs, keys, weights_dict=None):
        # We completely ignore the internal default and inject our grid-search weights
        return original_combine_cost_functions(costs, keys, weights_dict=custom_weights)
    
    # Monkey-patch the module to use our wrapped function
    sync_module.combine_cost_functions = mocked_combine_cost_functions
    
    try:
        # Run the sync process normally - it will now use your custom weights!
        game.synchronise_tracking_and_event_data()
        
        # Calculate your performance metrics (e.g., mean certainty)
        mean_certainty = game.tracking_data['sync_certainty'].mean()
        variance_certainty = game.tracking_data['sync_certainty'].var()
        
        # Store results
        result_entry = custom_weights.copy()
        result_entry["mean_certainty"] = mean_certainty
        result_entry["variance_certainty"] = variance_certainty
        results.append(result_entry)
        
    except Exception as e:
        print(f"Iteration {i+1} failed with error: {e}")
        
# --- STEP 3: Clean up and Restore original behavior ---
sync_module.combine_cost_functions = original_combine_cost_functions
print("\n--- GRID SEARCH COMPLETED & ORIGINAL FUNCTION RESTORED ---")

# Convert results to a DataFrame for easy sorting
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="mean_certainty", ascending=False)
print(df_results.head())